In [52]:
import pandas as pd
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE

In [34]:
df = pd.read_csv('PCA_26개컬럼.csv')

In [35]:
df = df[df['Segment'].isin(['A', 'B', 'C', 'D'])]
df

,C_PC1,C_PC2,C_PC3,이용금액_R3M_신용체크,_2순위카드이용금액,_1순위업종_이용금액,정상입금원금_B5M,이용금액_오프라인_B0M,_2순위업종_이용금액,최대이용금액_일시불_R12M,...,쇼핑_도소매_이용금액,이용금액_오프라인_R6M,청구금액_B0,청구금액_R6M,평잔_일시불_3M,잔액_일시불_B0M,Segment,PC1,PC2,PC3
0,-0.491893,0.425668,-0.348744,196,0,1928,9205,4043,1408,4906,...,0,11097,12226,88693,1791,998,D,-0.468294,-1.370911,0.571536
2,2.265364,0.811505,0.565863,23988,0,16924,16949,4524,1539,10112,...,1038,29192,21866,165221,6796,5312,C,-0.879638,1.602947,0.519022
3,0.599861,1.752421,0.017711,3904,0,2405,8418,3975,2284,3075,...,0,18056,16356,127371,772,730,D,0.269127,-1.316032,0.636893
8,8.382022,-2.295570,-0.828144,124967,38051,88663,13341,16045,12427,47166,...,2162,138508,20512,94241,28376,19144,C,1.218644,6.709643,-0.314953
10,0.835809,2.207009,2.077724,21001,0,8886,0,2437,4751,6681,...,0,14580,22512,35723,20399,20208,D,-0.628135,1.776798,-1.600463
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399979,4.541547,3.692684,2.115965,31187,4006,9379,8887,10581,2670,11317,...,1404,65522,11817,74153,4738,4390,D,-0.848922,1.366586,0.543775
2399987,4.030107,2.689812,-1.163825,42492,0,12882,9396,11488,3673,14037,...,2934,71653,17859,132623,3609,4618,C,-0.612759,-0.605555,0.859452
2399993,1.674691,-0.479688,-0.915211,72348,31303,11351,8912,12819,10561,20955,...,1183,74586,10810,72474,15212,10290,C,-0.738701,0.433571,0.698844
2399996,2.272736,-0.976253,-0.507896,27636,0,5608,21831,4676,1810,47684,...,1496,60373,14402,99849,9424,3351,D,-0.858109,1.430537,0.541463


In [36]:
# 클래스별로 분리
df_A = df[df['Segment'] == 'A']
df_B = df[df['Segment'] == 'B']
df_C = df[df['Segment'] == 'C']
df_D = df[df['Segment'] == 'D']

In [46]:
# C와 D만 A+B 전체 수만큼 다운샘플링
ab_total = len(df_A) + len(df_B)
cd_total = int(ab_total * 1.5)
cd_target_each = cd_total // 2

df_C_down = resample(df_C, replace=False, n_samples=cd_target_each, random_state=42)
df_D_down = resample(df_D, replace=False, n_samples=cd_target_each, random_state=42)

In [48]:
# 병합
df_balanced = pd.concat([df_A, df_B, df_C_down, df_D_down])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [50]:
# 결과 확인
print(df_balanced['Segment'].value_counts())

Segment
A    972
D    837
C    837
B    144
Name: count, dtype: int64


In [60]:
X = df_balanced.drop(columns=['Segment'])
y = df_balanced['Segment']

In [62]:
# B만 800개까지 증식
smote = SMOTE(sampling_strategy={'B': 800}, random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

In [64]:
# 다시 합쳐서 데이터프레임 만들기
df_final = pd.concat([pd.DataFrame(X_resampled, columns=X.columns),
                      pd.Series(y_resampled, name='Segment')], axis=1)

# 결과 클래스별 개수 확인
print(df_final['Segment'].value_counts())

Segment
A    972
D    837
C    837
B    800
Name: count, dtype: int64


In [66]:
df_final.to_csv('ABCD_26_up_downsampling.csv', index= False)

In [68]:
a = pd.read_csv('ABCD_26_up_downsampling.csv')
a

,C_PC1,C_PC2,C_PC3,이용금액_R3M_신용체크,_2순위카드이용금액,_1순위업종_이용금액,정상입금원금_B5M,이용금액_오프라인_B0M,_2순위업종_이용금액,최대이용금액_일시불_R12M,...,쇼핑_도소매_이용금액,이용금액_오프라인_R6M,청구금액_B0,청구금액_R6M,평잔_일시불_3M,잔액_일시불_B0M,PC1,PC2,PC3,Segment
0,3.340647,-3.744073,1.091390,55109,0,8000,5737,11449,7347,42407,...,1063,91070,8135,40345,10319,5305,-0.440716,2.401916,0.473536,D
1,5.388045,2.094659,0.197856,88038,25820,10008,10596,15407,9240,10109,...,2081,90047,24564,317682,4236,16410,14.469184,3.723127,1.293374,C
2,-1.030069,1.169899,-0.121168,0,0,0,1729,0,0,6696,...,0,0,585,5734,0,0,-0.693250,1.026248,-0.293011,C
3,-1.134224,-0.214285,0.233315,20648,3262,3824,3299,0,477,5238,...,0,2325,2480,20190,3480,1106,-0.647173,-0.338219,0.828986,D
4,-0.079700,-0.939331,0.391376,8459,0,3260,3000,2356,1773,10012,...,664,15350,2348,17975,2584,1419,-0.701052,0.115996,0.752728,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,5.402661,7.239230,2.723429,39728,957,5592,0,13054,1348,13711,...,784,44537,16392,100246,3964,2536,5.677299,2.734663,-0.173722,B
3442,5.211932,4.145780,1.159995,14698,0,4869,40878,4441,1238,5605,...,1392,20202,54983,406820,1563,2579,-0.845464,1.330554,0.554507,B
3443,4.934124,5.056856,2.439140,38110,6814,6614,11433,4647,1433,10506,...,1196,28854,38089,263328,3273,1363,2.247560,1.740677,-0.924084,B
3444,14.634794,7.749286,1.876457,124243,28492,98138,35366,15783,17401,61762,...,3927,112034,36951,234273,20305,16597,-1.414711,6.108551,-0.225881,B


In [70]:
a['Segment'].value_counts()

Segment
A    972
D    837
C    837
B    800
Name: count, dtype: int64